In [1]:
import keras
from keras.models import Model, load_model, Sequential
from keras import initializers, metrics
from keras.layers import Embedding, Dense, Input, Flatten, Dropout, Activation, Lambda, LSTM, TimeDistributed, Concatenate, Reshape
#from tensorflow.keras.layers.normalization import BatchNormalization
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import GlobalMaxPooling1D, GlobalAveragePooling1D
from keras.layers.merge import concatenate, add
#from tensorflow.python.keras.engine import Layer
from tensorflow.keras.layers import Layer
from keras.layers.wrappers import TimeDistributed
from keras import backend as K
from keras.utils.generic_utils import get_custom_objects
from tensorflow.keras.optimizers import Adam, SGD
#from keras.layers.merge import multiple, concatenate
from tensorflow.keras.layers import multiply, concatenate
from keras.regularizers import l2
from keras.callbacks import EarlyStopping
from keras.constraints import Constraint
import tensorflow as tf
from tensorflow.python.keras.backend import eager_learning_phase_scope
import json
from keras.models import model_from_json
from keras.utils.generic_utils import get_custom_objects

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

!pip install -q yfinance  yahoofinancials
import yfinance as yf
from yahoofinancials import YahooFinancials

In [ ]:

seed_value = 0
import os
os.environ['PYTHONHASHSEED'] = str(seed_value)
import random
random.seed(seed_value)
import numpy as np
np.random.seed(seed_value)
import tensorflow as tf
tf.random.set_seed(seed_value)

In [ ]:

tickers_list = ['SPY', 'IWM'] # 'SPY', 'IWM', 'IUSG', 'IUSV', 'EFA', 'EEM', 'DJP', 'IYR', 'JNK', 'IYR', 'LQD', ' TLT', 'TIP', 'IEF', 'AGG'
start_date = '2010-01-01'
end_date_str = '2022-01-01'
#data = yf.download(tickers_list, start_date, '2021-02-28')
yahoo_financials = YahooFinancials(tickers_list)
data_raw = yahoo_financials.get_historical_price_data(start_date=start_date, end_date=end_date_str, time_interval='daily')
#HSI_df = pd.DataFrame(data['^HSI']['prices'])

In [ ]:
data_raw

{'IWM': {'currency': 'USD',
  'eventsData': {'dividends': {'2010-03-24': {'amount': 0.172,
     'date': 1269437400,
     'formatted_date': '2010-03-24'},
    '2010-07-02': {'amount': 0.199,
     'date': 1278077400,
     'formatted_date': '2010-07-02'},
    '2010-09-23': {'amount': 0.165,
     'date': 1285248600,
     'formatted_date': '2010-09-23'},
    '2010-12-22': {'amount': 0.357,
     'date': 1293028200,
     'formatted_date': '2010-12-22'},
    '2011-03-24': {'amount': 0.169,
     'date': 1300973400,
     'formatted_date': '2011-03-24'},
    '2011-07-05': {'amount': 0.252,
     'date': 1309872600,
     'formatted_date': '2011-07-05'},
    '2011-09-23': {'amount': 0.25,
     'date': 1316784600,
     'formatted_date': '2011-09-23'},
    '2011-12-22': {'amount': 0.36,
     'date': 1324564200,
     'formatted_date': '2011-12-22'},
    '2012-03-23': {'amount': 0.251,
     'date': 1332509400,
     'formatted_date': '2012-03-23'},
    '2012-06-26': {'amount': 0.377,
     'date': 1340717

In [ ]:
for t in range(len(tickers_list)):
  i = tickers_list[t]
  cur_data = pd.DataFrame(data_raw[i]['prices'])
  cur_data = cur_data[['formatted_date', 'adjclose']]
  cur_data = pd.DataFrame(cur_data['adjclose'].values, index=cur_data['formatted_date'])
  cur_data.columns = [i]

  if t == 0:
    data = cur_data.copy()
  else:
    data = pd.concat([data, cur_data], axis=1)

  data = data.fillna(method='ffill')


In [ ]:
rtn_cols = tickers_list
rtn_cols_names = [s + '_rtn' for s in rtn_cols]

In [ ]:
rtn_cols, rtn_cols_names

(['SPY', 'IWM'], ['SPY_rtn', 'IWM_rtn'])

In [ ]:
test = data
test[rtn_cols_names] = test[rtn_cols].pct_change()
test = test[1::]
rtn = test[rtn_cols_names]
#test.head()
#data.shape[0]

In [ ]:
num_fund = len(tickers_list)
window = 50
y_window = 50


In [ ]:
tbl = []
y = []

for i in range(test.shape[0]-window-y_window):
  dat = test.values[i+1:i+window+1]
  y.append(test.values[i+window+1 : i+window+1+y_window, num_fund: 2*num_fund])
  tbl.append(dat)

tbl = np.stack(tbl)
y = np.stack(y)

print(y.shape)
print(tbl.shape)

(2920, 50, 2)
(2920, 50, 4)


In [ ]:


test_start = '2016-01-01'
cut_num = -test.loc[test_start:].shape[0]

x_train = tbl[:cut_num]
y_train = y[:cut_num]
x_test = tbl[cut_num::]
y_test = y[cut_num::]
print(tbl.shape)
print(x_train.shape)
print(x_test.shape)

num_features = x_train.shape[-1]

(2920, 50, 4)
(1409, 50, 4)
(1511, 50, 4)


In [ ]:
#### adding train data for 2 years for each model, and keep the testing data length for 2 years at the same time
trading_days = 252

years_retrain = 2
years_test = 5
import math
number_models = math.ceil(years_test / years_retrain)
print("Number of train/test pairs: ", number_models)

Number of train/test pairs:  3


In [ ]:

x_train_sets, y_train_sets, x_test_sets, y_test_sets = [], [], [], []
x_train_sets.append(x_train)
y_train_sets.append(y_train)
x_test_sets.append(x_test[0:trading_days * years_retrain])
y_test_sets.append(y_test[0:trading_days * years_retrain])

for i in range(1, number_models):
  x_train_sets.append(np.concatenate((x_train_sets[i - 1], x_test_sets[i - 1])))
  y_train_sets.append(np.concatenate((y_train_sets[i - 1], y_test_sets[i - 1])))
  x_test_sets.append(x_test[i * trading_days * years_retrain : (i + 1) * trading_days * years_retrain])
  y_test_sets.append(y_test[i * trading_days * years_retrain : (i + 1) * trading_days * years_retrain])

for i in range(len(x_train_sets)):
  print('Train/Test pair: ', i)
  print(x_train_sets[i].shape)
  print(y_train_sets[i].shape)
  print(x_test_sets[i].shape)
  print(y_test_sets[i].shape)

Train/Test pair:  0
(1409, 50, 4)
(1409, 50, 2)
(504, 50, 4)
(504, 50, 2)
Train/Test pair:  1
(1913, 50, 4)
(1913, 50, 2)
(504, 50, 4)
(504, 50, 2)
Train/Test pair:  2
(2417, 50, 4)
(2417, 50, 2)
(503, 50, 4)
(503, 50, 2)


In [ ]:
# customize loss function
def custom_mse(y_true, y_pred):
  y_true_flat = K.batch_flatten(y_true)
  sharp_ratio_loss = []

  for i in range(y_window):
    product = K.sum(y_true_flat[:, i*num_fund:num_fund*(i+1)] * y_pred, axis=-1, keepdims=True)
    sharp_ratio_loss.append(K.exp(-1 * K.mean(product)) + K.exp( K.std(product) ))

  cumprod_product = K.sum((K.cumprod(1 + y_true, axis=1)[:, -1, :]) * y_pred, axis=-1, keepdims=True)
  cumprod_loss = K.exp( -1 * K.mean(cumprod_product) )
  final_loss = K.sum(sharp_ratio_loss) / y_window + 3 * cumprod_loss

  return final_loss


In [ ]:

class CustomStopper(keras.callbacks.EarlyStopping):
  def __init__(self, monitor='val_loss', min_delta=0, patience=0, verbose=0, mode='auto', start_epoch=40, restore_best_weights=False):
    super().__init__(monitor=monitor, min_delta=min_delta, patience=patience, verbose=verbose, mode=mode, restore_best_weights=restore_best_weights)
    self.start_epoch = start_epoch

  def on_epoch_end(self, epoch, logs=None):
    if epoch > self.start_epoch:
      super().on_epoch_end(epoch, logs)

def get_lstm_model():

  print('Build model...')
  model = Sequential()

  model.add(Input(shape=(window, num_features)))

  hidden = 64

  model.add(LSTM(hidden))

  model.add(BatchNormalization())

  model.add(Dense(num_fund))
  model.add(Activation('softmax'))

  model.compile(loss=custom_mse, optimizer='adam')

  return model


In [ ]:

early_stop_start_epoch = 30
max_epochs = 100
batch_size = 250

models = []

for i in range(len(x_train_sets)):

  X_train, X_val, y_train, y_val = train_test_split(x_train_sets[i], y_train_sets[i], test_size=0.1, random_state=42)

  early_stop = CustomStopper(monitor='val_loss', min_delta=0, patience=5, verbose=0, mode='min', start_epoch=early_stop_start_epoch, restore_best_weights=True)

  print("Train...")
  model = get_lstm_model()

  model.fit(X_train, y_train, batch_size=batch_size, epochs=max_epochs, validation_data=(X_val, y_val), callbacks=[early_stop])

  score = model.evaluate(x_test_sets[i], y_test_sets[i], batch_size=batch_size)
  print('Test score: ', score)

  models.append(model)

Train...
Build model...
Epoch 1/100
6/6 [==============================] - 15s 322ms/step - loss: 3.0858 - val_loss: 3.0928
Epoch 2/100
6/6 [==============================] - 0s 64ms/step - loss: 3.0834 - val_loss: 3.0922
Epoch 3/100
6/6 [==============================] - 0s 61ms/step - loss: 3.0825 - val_loss: 3.0913
Epoch 4/100
6/6 [==============================] - 0s 63ms/step - loss: 3.0822 - val_loss: 3.0909
Epoch 5/100
6/6 [==============================] - 0s 61ms/step - loss: 3.0821 - val_loss: 3.0908
Epoch 6/100
6/6 [==============================] - 0s 64ms/step - loss: 3.0819 - val_loss: 3.0909
Epoch 7/100
6/6 [==============================] - 0s 62ms/step - loss: 3.0817 - val_loss: 3.0908
Epoch 8/100
6/6 [==============================] - 0s 61ms/step - loss: 3.0814 - val_loss: 3.0904
Epoch 9/100
6/6 [==============================] - 0s 64ms/step - loss: 3.0811 - val_loss: 3.0900
Epoch 10/100
6/6 [==============================] - 0s 61ms/step - loss: 3.0812 - val_loss: 

In [ ]:

outputs = []
rtn_deep_list = []
static_model = models[0]
for i, model in enumerate(models):
  pred = model.predict(x_test_sets[i])
  outputs.append(pred)
  rtn_deep_list.append(np.diag(np.dot(y_test_sets[i][:, 0, :], pred.transpose())))

outputs = np.concatenate(outputs, axis=0)
rtn_deep = np.concatenate(rtn_deep_list)

#pred_weight = pd.DataFrame(outputs, index=rtn[cut_num - y_window + 1: -y_window + 1].index, columns = data.columns[:num_fund])


In [ ]:
outputs

array([[4.9749973e-01, 5.0250024e-01],
       [4.2811394e-01, 5.7188606e-01],
       [3.9602780e-01, 6.0397226e-01],
       ...,
       [9.9996126e-01, 3.8779377e-05],
       [9.9995732e-01, 4.2629901e-05],
       [9.9996555e-01, 3.4474273e-05]], dtype=float32)